In [ ]:
import datasets

In [ ]:
from datasets import load_dataset

# full dataset
ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")

# dataset streaming (will only download the data as needed)
# ds = load_dataset("bigcode/bigcodebench", streaming=True, split="v0.1.4")



In [ ]:
print(ds)


In [ ]:
iter = ds.__iter__()

sample = next(iter)

for key, value in sample.items():
    print(f"############# {key} #############")
    print(f"{value}")
    # print("#################################")


In [ ]:
code = sample['complete_prompt'] + sample['canonical_solution']


In [ ]:
delimeter = "class TestCases(unittest.TestCase):"
run_code = """if __name__ == '__main__':
    unittest.main()
"""
test_code = sample['test'].split(delimeter)
test_code = f"{test_code[0]}\n{code}\n{delimeter}\n{test_code[1]}\n{run_code}"

In [ ]:
for i, sample in enumerate(ds):
    if delimeter not in sample['test']:
        print(f"Sample {i} does not contain the delimiter.")
        for key, value in sample.items():
            print(f"############# {key} #############")
            print(f"{value}")

In [ ]:
test_code_template = """
# 2. The function to run tests and calculate the ratio
def run_tests_and_get_ratio():
    # Create a TestLoader instance
    loader = unittest.TestLoader()

    # Create a TestSuite
    suite = unittest.TestSuite()
    
    # Add tests from the test class to the suite
    suite.addTests(loader.loadTestsFromTestCase(TestCases))

    # Create a TestRunner
    # We use a TextTestRunner with verbosity=0 to suppress the default output,
    # and stream=sys.stderr to prevent cluttering stdout. You can adjust this.
    runner = unittest.TextTestRunner(verbosity=0, stream=sys.stderr)
    
    # Run the tests. The run() method returns a TestResult object.
    # print("Running tests...")
    result = runner.run(suite)
    # print("Tests finished.")
    
    # --- The Calculation Part ---
    
    # Total number of tests that were run
    total_run = result.testsRun
    
    # Number of failures and errors
    failures = len(result.failures)
    errors = len(result.errors)
    
    # Calculate the number of passed tests
    passed = total_run - (failures + errors)

    # Calculate the ratio
    if total_run > 0:
        pass_ratio = passed / total_run
    else:
        pass_ratio = 0.0

    return pass_ratio, result.wasSuccessful()


# 3. Main execution block, replacing the simple unittest.main()
if __name__ == '__main__':
    import unittest
    import sys

    ratio, was_successful = run_tests_and_get_ratio()
    sys.exit(int(ratio * 100))
"""

In [ ]:


def create_test_code(sample):
    code = sample['complete_prompt'] + sample['canonical_solution']
    delimeter = "class TestCases(unittest.TestCase):"
    
    # test_code = sample['test'].split(delimeter, maxsplit=1)
    # test_code = f"{test_code[0]}\n{code}\n{delimeter}\n{test_code[1]}\n{test_code_template}"
    test_code = f"{sample['test']}\n{code}\n{test_code_template}"
    return test_code

In [ ]:
from Server.code_client import CodeClient, ExecutionResult

client = CodeClient(
    port=8002,
    
)

In [ ]:
client.health_check()

In [ ]:
client.get_server_info()

In [ ]:
import asyncio

async def execute_code(code: str, execution_timeout=10) -> ExecutionResult:
    """
    Execute the given code and return the result.
    """
    result = client.execute_code(code=code, execution_timeout=execution_timeout)
    return result

test_cases = [
    execute_code("import time\ntime.sleep(5)\nprint('Hello, World1!')"),
    execute_code("print('Hello, World2!')"),
]
# result1 = await (execute_code("import time\ntime.sleep(5)\nprint('Hello, World1!')"))
# result2 = await (execute_code("print('Hello, World2!')"))
test_results = await asyncio.gather(*test_cases)

for i, res in enumerate(test_results):
    print(f"result[{res.execution_time}]: {res.stdout}")
    



In [ ]:
res = client.execute_code(code="import time\ntime.sleep(5)\nprint('Hello, World!')")
print(f"Execution Result:\n {res}")

In [ ]:
res = client.execute_code(code="import time\ntime.sleep(20)\nprint('Hello, World?')")
print(f"Execution Result:\n {res}")

In [ ]:
res = client.execute_code(code="print('Hello, World?')")
print(f"Execution Result:\n {res}")

In [ ]:
n_tasks = 1

results = []
for id, sample in zip(range(n_tasks), ds):
    print(f"########## Sample {id} #############")
    test_code = create_test_code(sample)
    res:ExecutionResult = client.execute_code(code=test_code, execution_timeout=15)
    results.append(res)
    
    print(f"Suceess[{res.execution_time}]: {res.success}")
    if not res.success:
        # print(f"Error type: {res.error_type}")
        for key, value in res.__dict__.items():
            print(f"{key}:\n {value}")

In [ ]:
print(results[0])